### Imports
Core libraries for data handling, API calls, and model inference.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from googleapiclient.discovery import build
from transformers import pipeline
from dotenv import load_dotenv
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta
import glob
import re
import sys
from langdetect import detect, DetectorFactory

### API config
Load credentials and initialize the YouTube client.

In [ ]:
load_dotenv()

API_KEY = os.getenv('YOUTUBE_API_KEY') 
SEARCH_QUERY = 'AI'

DEFAULT_START_DATE = datetime(2022, 1, 1, tzinfo=timezone.utc)
END_DATE = datetime(2025, 12, 31, tzinfo=timezone.utc)
TARGET_CATEGORY = "28"

if not API_KEY:
    print("WARNING: API Key not found. Please check your .env file.")
else:
    print("Configuration loaded.")
    youtube = build('youtube', 'v3', developerKey=API_KEY)
    print("Connected to YouTube API successfully.")

### Fetch
Pull monthly videos/comments from the API and persist resumable checkpoints.

In [ ]:
OUTPUT_DIR = "youtube_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Search filters
SEARCH_QUERY = 'AI -music -song -papai -dance' 
TARGET_CATEGORY = "28" # Science & Technology

master_csv_path = os.path.join(OUTPUT_DIR, "master_comments_raw.csv")

all_comments_data = []
current_date = DEFAULT_START_DATE

print("Start fetch")

# Checkpoint resume
completed_months = []
summary_pattern = os.path.join(OUTPUT_DIR, "summary_*.txt")
for file_path in glob.glob(summary_pattern):
    match = re.search(r"summary_(\d{4}-\d{2})\.txt", file_path)
    if match:
        completed_months.append(match.group(1))

if completed_months:
    completed_months.sort()
    last_completed_str = completed_months[-1]
    last_completed_date = datetime.strptime(last_completed_str, "%Y-%m")
    
    current_date = (last_completed_date + relativedelta(months=1)).replace(tzinfo=timezone.utc)
    print(f"Resume from {last_completed_str}")
    
    if os.path.exists(master_csv_path):
        try:
            all_comments_data = pd.read_csv(master_csv_path).to_dict(orient="records")
            print(f"Loaded {len(all_comments_data):,} rows")
        except Exception as e:
            print(f"CSV read failed ({e}). Starting empty")
else:
    print("No checkpoints found")

print(f"Start month: {current_date.strftime('%Y-%m')}")

# Main loop
while current_date <= END_DATE:
    month_label = current_date.strftime('%Y-%m')
    next_month = current_date + relativedelta(months=1)
    
    print(f"Month {month_label}")
    
    raw_video_ids = []
    search_page_token = None
    search_page_count = 0
    
    # 1) Fetch video IDs
    while True:
        try:
            search_page_count += 1
            search_request = youtube.search().list(
                part="id",
                q=SEARCH_QUERY,
                type="video",
                videoCategoryId=TARGET_CATEGORY,
                relevanceLanguage="en", # Language filter
                publishedAfter=current_date.isoformat(),
                publishedBefore=next_month.isoformat(),
                maxResults=50,
                pageToken=search_page_token
            )
            search_response = search_request.execute()
            
            items = search_response.get("items", [])
            for item in items:
                if item["id"].get("videoId"):
                    raw_video_ids.append(item["id"]["videoId"])
                    
            search_page_token = search_response.get("nextPageToken")
            if not search_page_token:
                break
                
        except Exception as e:
            error_msg = str(e).lower()
            if any(keyword in error_msg for keyword in ["quota", "429", "403", "400", "expired"]):
                # Stop early to preserve checkpoint state.
                print("Critical API error. Stopping to keep checkpoints")
                print(f"Details: {e}")
                sys.exit() 
            print(f"Search page {search_page_count} failed: {e}")
            break
            
    total_videos_found = len(raw_video_ids)
    print(f"Videos: {total_videos_found} ({search_page_count} pages)")
    
    if not raw_video_ids:
        summary_filename = os.path.join(OUTPUT_DIR, f"summary_{month_label}.txt")
        with open(summary_filename, "w", encoding="utf-8") as f:
            f.write(f"YOUTUBE ANALYSIS REPORT FOR MONTH: {month_label}\n")
            f.write(f"No videos found for this period under the defined criteria.\n")
        
        print("No videos. Skipped")
        current_date = next_month
        continue
        
    # 2) Fetch video stats
    video_metadata_list = []
    print("Fetching video stats...")
    
    for i in range(0, len(raw_video_ids), 50):
        batch_ids = raw_video_ids[i:i+50]
        try:
            stats_request = youtube.videos().list(
                part="snippet,statistics",
                id=",".join(batch_ids)
            )
            stats_response = stats_request.execute()
            
            for item in stats_response.get("items", []):
                v_id = item["id"]
                title = item["snippet"]["title"]
                stats = item.get("statistics", {})
                
                video_metadata_list.append({
                    "id": v_id, 
                    "title": title, 
                    "views": int(stats.get("viewCount", 0)),
                    "likes": int(stats.get("likeCount", 0)),
                    "comments": int(stats.get("commentCount", 0))
                })
        except Exception as e:
            error_msg = str(e).lower()
            if any(keyword in error_msg for keyword in ["quota", "429", "403", "400", "expired"]):
                print("Critical API error. Stopping to keep checkpoints")
                sys.exit()
            print(f"Stats batch failed: {e}")

    video_metadata_df = pd.DataFrame(video_metadata_list)
        
    # 3) Fetch comments
    month_comments_count = 0
    print("Fetching comments...")
    
    for video_id in raw_video_ids:
        comment_page_token = None
        
        while True:
            try:
                comment_request = youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    maxResults=100,
                    pageToken=comment_page_token,
                    textFormat="plainText"
                )
                comment_response = comment_request.execute()
                
                items = comment_response.get("items", [])
                for item in items:
                    reply_count = item["snippet"].get("totalReplyCount", 0)
                    comment_snippet = item["snippet"]["topLevelComment"]["snippet"]
                    
                    all_comments_data.append({
                        "Month_Video_Published": month_label,
                        "Video_ID": video_id,
                        "Comment_Date": comment_snippet["publishedAt"],
                        "Author": comment_snippet.get("authorDisplayName", "Unknown"),
                        "Likes": comment_snippet.get("likeCount", 0),
                        "Reply_Count": reply_count,
                        "Comment": comment_snippet["textDisplay"]
                    })
                    month_comments_count += 1
                    
                comment_page_token = comment_response.get("nextPageToken")
                if not comment_page_token:
                    break
                    
            except Exception as e:
                # Comments disabled or request failed
                break
                
    # 4) Write monthly summary
    summary_filename = os.path.join(OUTPUT_DIR, f"summary_{month_label}.txt")
    with open(summary_filename, "w", encoding="utf-8") as f:
        f.write(f"==================================================\n")
        f.write(f"MONTHLY REPORT - {month_label}\n")
        f.write(f"==================================================\n\n")
        f.write(f"Total Videos Discovered: {total_videos_found:,}\n")
        f.write(f"Total Comments Processed: {month_comments_count:,}\n\n")
        
        if not video_metadata_df.empty:
            f.write(f"Total Network Views: {video_metadata_df['views'].sum():,}\n")
            f.write(f"Total Network Likes: {video_metadata_df['likes'].sum():,}\n\n")
            
            f.write(f"--- TOP 5 MOST COMMENTED VIDEOS (Highest Debate) ---\n")
            top_comments = video_metadata_df.sort_values(by="comments", ascending=False).head(5)
            for rank, (_, row) in enumerate(top_comments.iterrows(), start=1):
                f.write(f"{rank}. {row['title']}\n")
                f.write(f"   ID: {row['id']} | Comments: {row['comments']:,}\n\n")
                
            f.write(f"--- TOP 5 MOST VIEWED VIDEOS (Highest Reach) ---\n")
            top_views = video_metadata_df.sort_values(by="views", ascending=False).head(5)
            for rank, (_, row) in enumerate(top_views.iterrows(), start=1):
                f.write(f"{rank}. {row['title']}\n")
                f.write(f"   ID: {row['id']} | Views: {row['views']:,}\n\n")
        else:
            f.write("No engagement statistics available.\n")
            
    print(f"Saved {summary_filename}")
    
    # Persist progress each month.
    master_df = pd.DataFrame(all_comments_data)
    master_df.to_csv(master_csv_path, index=False, encoding="utf-8")
    
    print(f"Added {month_comments_count:,} rows | Total {len(master_df):,}")
    print("-")
    
    current_date = next_month

print("Done. Database updated")

### Clean
Normalize text, keep English comments, and write a clean base dataset.

In [ ]:
# Keep langdetect deterministic
DetectorFactory.seed = 0

print("Load raw data")
df = pd.read_csv("youtube_data/comments_raw.csv")
original_count = len(df)
print(f"Rows in: {original_count:,}")

# 1) Drop empty and exact duplicate comments
df = df.dropna(subset=['Comment'])
df = df.drop_duplicates(subset=['Video_ID', 'Comment', 'Author'])
print(f"Removed empty/dupes: {original_count - len(df):,}")

# 2) Normalize text
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www.\S+', '', text)  # Remove links
    text = re.sub(r'@\w+', '', text)             # Remove mentions
    text = re.sub(r'\s+', ' ', text).strip()     # Normalize spaces
    return text

print("Clean text")
df['Clean_Comment'] = df['Comment'].apply(clean_text)
df = df[df['Clean_Comment'].str.len() > 0]

# 3) Keep English only
def is_english(text):
    try:
        if len(text) < 10: 
            return False
        return detect(text) == 'en'
    except:
        return False

print("Filter language")
df['Is_English'] = df['Clean_Comment'].apply(is_english)
df = df[df['Is_English'] == True]

# 4) Cap text length
df['Clean_Comment'] = df['Clean_Comment'].str.slice(0, 500)

# 5) Save cleaned dataset
final_columns = ['Month_Video_Published', 'Video_ID', 'Comment_Date', 'Author', 'Likes', 'Reply_Count', 'Clean_Comment']
existing_columns = [col for col in final_columns if col in df.columns]
df_final = df[existing_columns]

output_filename = "youtube_data/comments_cleaned.csv"
df_final.to_csv(output_filename, index=False, encoding='utf-8')

print("Done")
print(f"Rows out: {len(df_final):,}")
print(f"Removed total: {original_count - len(df_final):,}")
print(f"Saved: {output_filename}")

### Emotions
Run emotion inference in chunks with resume support via output checkpoints.

In [ ]:
import pandas as pd
import torch
import gc
import os
from transformers import pipeline
from tqdm.auto import tqdm

if torch.cuda.is_available():
    device_id = 0
    print("Device: CUDA")
elif torch.backends.mps.is_available():
    device_id = "mps"
    print("Device: MPS")
else:
    device_id = -1
    print("Device: CPU")

INPUT_PATH = "youtube_data/comments_cleaned.csv"
OUTPUT_PATH = "youtube_data/comments_emotions.csv"
MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"

BATCH_SIZE = 64
CHUNK_SIZE = 10000

print(f"Load: {INPUT_PATH}")
df = pd.read_csv(INPUT_PATH)

if 'Emotion' not in df.columns:
    df['Emotion'] = None
if 'Emotion_Confidence' not in df.columns:
    df['Emotion_Confidence'] = None

start_idx = df['Emotion'].isnull().isna().sum()
if os.path.exists(OUTPUT_PATH):
    print("Resume check")
    df_existing = pd.read_csv(OUTPUT_PATH)
    processed_count = df_existing['Emotion'].notna().sum()
    if processed_count > 0:
        df.iloc[:processed_count] = df_existing.iloc[:processed_count]
        start_idx = processed_count
        print(f"Resume at row: {start_idx:,}")

text_list = df['Clean_Comment'].astype(str).tolist()

print(f"Load model: {MODEL_NAME}")
classifier = pipeline(
    "text-classification", 
    model=MODEL_NAME, 
    device=device_id, 
    return_all_scores=False
)

print(f"Rows left: {len(text_list) - start_idx:,}")

for chunk_start in tqdm(range(start_idx, len(text_list), CHUNK_SIZE), desc="Progress"):
    chunk_end = min(chunk_start + CHUNK_SIZE, len(text_list))
    chunk_texts = text_list[chunk_start:chunk_end]
    
    chunk_emotions = []
    chunk_scores = []
    
    results = classifier(
        chunk_texts, 
        batch_size=BATCH_SIZE, 
        truncation=True, 
        max_length=128
    )
    
    for out in results:
        chunk_emotions.append(out['label'])
        chunk_scores.append(out['score'])
    
    df.iloc[chunk_start:chunk_end, df.columns.get_loc('Emotion')] = chunk_emotions
    df.iloc[chunk_start:chunk_end, df.columns.get_loc('Emotion_Confidence')] = chunk_scores
    
    df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
    
    if device_id == "mps":
        torch.mps.empty_cache()
    gc.collect()

print("Done")
print(f"Saved: {OUTPUT_PATH}")

### Filter terms
Keep rows that pass confidence threshold and contain target AI keywords.

In [ ]:
INPUT_PATH = "youtube_data/comments_emotions.csv"
OUTPUT_PATH = "youtube_data/comments_terms.csv"

print(f"Load: {INPUT_PATH}")
df = pd.read_csv(INPUT_PATH)
initial_count = len(df)

CONFIDENCE_THRESHOLD = 0.4
print(f"Confidence >= {CONFIDENCE_THRESHOLD}")

df_filtered = df[df['Emotion_Confidence'] >= CONFIDENCE_THRESHOLD]
after_confidence_count = len(df_filtered)

KEYWORDS = ['AI', 'AI Bubble', 'ChatGPT', 'artificial intelligence', 'Gemini']
keyword_pattern = '|'.join(KEYWORDS)

print("Apply keyword filter")
keyword_mask = df_filtered['Clean_Comment'].str.contains(keyword_pattern, case=False, na=False)
df_filtered = df_filtered[keyword_mask]
final_count = len(df_filtered)

df_filtered.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

rows_dropped_confidence = initial_count - after_confidence_count
rows_dropped_keywords = after_confidence_count - final_count
rows_dropped_total = initial_count - final_count
percent_dropped = (rows_dropped_total / initial_count) * 100

print("Done")
print(f"Rows in: {initial_count:,}")
print(f"After confidence: {after_confidence_count:,} (-{rows_dropped_confidence:,})")
print(f"After keywords: {final_count:,} (-{rows_dropped_keywords:,})")
print(f"Dropped total: {rows_dropped_total:,} ({percent_dropped:.2f}%)")
print(f"Saved: {OUTPUT_PATH}")

### FinBERT
Run financial sentiment inference in chunks and save progress safely.

In [ ]:
import pandas as pd
import torch
import time
import os
import gc
from transformers import pipeline
from tqdm.auto import tqdm

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")

INPUT_PATH = "youtube_data/comments_terms.csv"
PROGRESS_PATH = "youtube_data/comments_finbert_progress.csv"
OUTPUT_PATH = "youtube_data/comments_finbert_sentiment.csv"

print(f"Load: {INPUT_PATH}")
df = pd.read_csv(INPUT_PATH, low_memory=False)

if 'FinBERT_Sentiment' not in df.columns:
    df['FinBERT_Sentiment'] = None
if 'FinBERT_Confidence' not in df.columns:
    df['FinBERT_Confidence'] = None

start_idx = 0

if os.path.exists(PROGRESS_PATH):
    print("Resume check")
    df_progress = pd.read_csv(PROGRESS_PATH, low_memory=False)
    processed_count = df_progress['FinBERT_Sentiment'].notna().sum()
    
    if processed_count > 0:
        df.iloc[:processed_count] = df_progress.iloc[:processed_count]
        start_idx = processed_count
        print(f"Resume at row: {start_idx:,}")

comments_list = df['Clean_Comment'].fillna("").astype(str).tolist()
total_rows = len(comments_list)

if start_idx >= total_rows:
    print("Nothing to run")
    exit()

print("Load model: ProsusAI/finbert")
sentiment_pipeline = pipeline(
    "text-classification", 
    model="ProsusAI/finbert", 
    device=device
)

BATCH_SIZE = 32
CHUNK_SIZE = 2048

remaining_rows = total_rows - start_idx
print(f"Rows left: {remaining_rows:,}")
start_time = time.time()

for chunk_start in tqdm(range(start_idx, total_rows, CHUNK_SIZE), desc="Progress"):
    chunk_end = min(chunk_start + CHUNK_SIZE, total_rows)
    chunk_texts = comments_list[chunk_start:chunk_end]
    
    results = sentiment_pipeline(
        chunk_texts, 
        batch_size=BATCH_SIZE, 
        truncation=True, 
        max_length=128
    )
    
    chunk_labels = [out['label'] for out in results]
    chunk_scores = [out['score'] for out in results]
        
    df.iloc[chunk_start:chunk_end, df.columns.get_loc('FinBERT_Sentiment')] = chunk_labels
    df.iloc[chunk_start:chunk_end, df.columns.get_loc('FinBERT_Confidence')] = chunk_scores
    
    # Save progress after each chunk.
    df.to_csv(PROGRESS_PATH, index=False, encoding='utf-8')
    
    if device.type == "mps":
        torch.mps.empty_cache()
    gc.collect()

end_time = time.time()
elapsed_time = end_time - start_time
rows_per_sec = remaining_rows / elapsed_time if elapsed_time > 0 else 0

print("Save final")
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

if os.path.exists(PROGRESS_PATH):
    os.remove(PROGRESS_PATH)

print("Done")
print(f"Processed: {remaining_rows:,}")
print(f"Time (min): {elapsed_time / 60:.2f}")
print(f"Rows/s: {rows_per_sec:.2f}")
print(df['FinBERT_Sentiment'].value_counts().to_string())
print(f"Saved: {OUTPUT_PATH}")

### Polarize
Keep only non-neutral, confident FinBERT predictions for final analysis.

In [ ]:
import pandas as pd

INPUT_PATH = "youtube_data/comments_finbert_sentiment.csv"
OUTPUT_PATH = "youtube_data/comments_finbert_polarized.csv"

print(f"Load: {INPUT_PATH}")
df = pd.read_csv(INPUT_PATH, low_memory=False)

total_original = len(df)

df['FinBERT_Sentiment'] = df['FinBERT_Sentiment'].astype(str).str.lower().str.strip()
df['FinBERT_Confidence'] = pd.to_numeric(df['FinBERT_Confidence'], errors='coerce')

# Keep only non-neutral FinBERT rows.
before_finbert = len(df)
df_filtered = df[df['FinBERT_Sentiment'] != 'neutral'].copy()
after_finbert = len(df_filtered)

before_finbert_conf = len(df_filtered)
df_filtered = df_filtered[df_filtered['FinBERT_Confidence'] >= 0.4]
after_finbert_conf = len(df_filtered)

df_polarized = df_filtered.copy()
total_final = len(df_polarized)
rows_removed = total_original - total_final
percent_removed = (rows_removed / total_original) * 100 if total_original > 0 else 0

df_polarized.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print("Done")
print(f"Rows in: {total_original:,}")
print(f"Removed FinBERT neutral: {before_finbert - after_finbert:,}")
print(f"Removed FinBERT conf < 0.4: {before_finbert_conf - after_finbert_conf:,}")
print(f"Rows out: {total_final:,}")
print(f"Dropped total: {rows_removed:,} ({percent_removed:.2f}%)")
print(f"Saved: {OUTPUT_PATH}")